# Upload Demand Fulfillment KPIs to `jkplanningV1.jkt_plan_kpis`

Reads the **Demand Fulfillment** sheet of the BTP PCR Curing plan workbook, parses the summary line on row 2, and inserts a single row into `jkt_plan_kpis`.

In [1]:
import re
from datetime import datetime
import openpyxl
import mysql.connector

In [2]:
# ---- DB connection config (fill in placeholders) ----
DB_HOST     = "35.208.174.2"
DB_PORT     = 3306
DB_USER     = "root"
DB_PASSWORD = "Dev112233"
DB_NAME     = "jkplanningV1"

# ---- Inputs ----
EXCEL_PATH = "BTP_PCR_Curing_LP_v4_PlanSchedule_May_2026-05-01_31V5Days.xlsx"
SHEET_NAME = "Demand Fulfillment"
PLAN_ID    = "1001"
CREATED_BY = "Algo8 AI"

In [4]:
# Read row 2 of the Demand Fulfillment sheet (single summary string).
wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb[SHEET_NAME]
summary = ws.cell(row=2, column=1).value
print("Summary line:\n", summary)

Summary line:
 Demand: 685,215  |  Planned: 680,027  |  Gap: 5,672  |  Fulfillment: 99.2%  |  Avg Util: 87.4%  |  Changeovers: 170  |  Mould Cleans: 66 | No of SKUs: 94


In [6]:
# Parse KPIs out of the summary string.
# Example: 'Demand: 685,215  |  Planned: 680,027  |  Gap: 5,672  |  Fulfillment: 99.2%  |  Avg Util: 87.4%  |  Changeovers: 170  |  Mould Cleans: 66 | No of SKUs: 94'

def _find(pattern, text):
    m = re.search(pattern, text)
    if not m:
        raise ValueError(f"Pattern not found in summary: {pattern}")
    return m.group(1)

fulfillment_pct = float(_find(r"Fulfillment:\s*([\d.]+)\s*%", summary))
avg_util_pct    = float(_find(r"Avg Util:\s*([\d.]+)\s*%", summary))
changeovers     = int(_find(r"Changeovers:\s*([\d,]+)", summary).replace(",", ""))
no_of_skus      = int(_find(r"No of SKUs:\s*([\d,]+)", summary).replace(",", ""))

row = {
    "plan_id":             PLAN_ID,
    "demandFulfillment":   fulfillment_pct,
    "demandSKU":           no_of_skus,
    "planSKU":             no_of_skus,
    "capacityUtilisation": avg_util_pct,
    "curingChangeovers":   changeovers,
    "createdAt":           datetime.now(),
    "createdBy":           CREATED_BY,
}
row

{'plan_id': '1001',
 'demandFulfillment': 99.2,
 'demandSKU': 94,
 'planSKU': 94,
 'capacityUtilisation': 87.4,
 'curingChangeovers': 170,
 'createdAt': datetime.datetime(2026, 5, 21, 14, 37, 39, 516701),
 'createdBy': 'Algo8 AI'}

In [7]:
# Insert into jkt_plan_kpis.
conn = mysql.connector.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)

insert_sql = """
    INSERT INTO jkt_plan_kpis
        (plan_id, demandFulfillment, demandSKU, planSKU,
         capacityUtilisation, curingChangeovers, createdAt, createdBy)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

values = (
    row["plan_id"],
    row["demandFulfillment"],
    row["demandSKU"],
    row["planSKU"],
    row["capacityUtilisation"],
    row["curingChangeovers"],
    row["createdAt"],
    row["createdBy"],
)

try:
    cur = conn.cursor()
    cur.execute(insert_sql, values)
    conn.commit()
    print(f"Inserted 1 row into jkt_plan_kpis. Row id: {cur.lastrowid}")
finally:
    cur.close()
    conn.close()

Inserted 1 row into jkt_plan_kpis. Row id: 0


In [8]:
# Verify the inserted row.
conn = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)
try:
    cur = conn.cursor(dictionary=True)
    cur.execute(
        "SELECT * FROM jkt_plan_kpis WHERE plan_id = %s ORDER BY createdAt DESC LIMIT 1",
        (PLAN_ID,),
    )
    print(cur.fetchone())
finally:
    cur.close()
    conn.close()

{'plan_id': '1001', 'demandFulfillment': 99.2, 'demandSKU': 94, 'planSKU': 94, 'capacityUtilisation': 87.4, 'curingChangeovers': 170, 'createdAt': datetime.datetime(2026, 5, 21, 14, 37, 40), 'createdBy': 'Algo8 AI'}


In [5]:
# Build the rows from the Shift Schedule sheet.
SHIFT_SHEET = "Shift Schedule"
ws = wb[SHIFT_SHEET]

now = datetime.now()
rows_to_insert = []

for r in range(4, ws.max_row + 1):
    date_v, shift_v, _machine, sku_code, start_t, end_t, qty, cycle, _gt_inv, remarks, sku_desc = (
        ws.cell(row=r, column=c).value for c in range(1, 12)
    )
    if all(v is None for v in (date_v, shift_v, sku_code, start_t, end_t, qty)):
        continue

    rows_to_insert.append((
        PLAN_ID,
        sku_code,
        sku_desc,
        date_v.date() if hasattr(date_v, "date") else date_v,
        shift_v,
        start_t,
        end_t,
        int(qty) if qty is not None else None,
        float(cycle) if cycle is not None else None,
        remarks,
        now,
        CREATED_BY,
    ))

print(f"Prepared {len(rows_to_insert)} rows for jkt_plan")
print("Sample:", rows_to_insert[0])


Prepared 14292 rows for jkt_plan
Sample: ('1001', 'CHANGEOVER', None, datetime.date(2026, 5, 8), 'B', datetime.datetime(2026, 5, 8, 21, 15), datetime.datetime(2026, 5, 9, 2, 15), 0, 0.0, 'C/O from 1325214613071TTMX0 to 1325214812074TUHL0', datetime.datetime(2026, 5, 21, 15, 4, 0, 29417), 'Algo8 AI')


In [6]:
# Bulk insert into jkt_plan.
conn = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)

insert_sql = """
    INSERT INTO jkt_plan
        (plan_id, skuCode, skuDescription, date, shift,
         startTime, endTime, qty, cycleTime, remarks, createdAt, createdBy)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

BATCH = 1000
try:
    cur = conn.cursor()
    total = 0
    for i in range(0, len(rows_to_insert), BATCH):
        chunk = rows_to_insert[i:i+BATCH]
        cur.executemany(insert_sql, chunk)
        total += cur.rowcount
        print(f"  inserted {total}/{len(rows_to_insert)}")
    conn.commit()
    print(f"Done. Inserted {total} rows into jkt_plan.")
finally:
    cur.close()
    conn.close()


  inserted 1000/14292
  inserted 2000/14292
  inserted 3000/14292
  inserted 4000/14292
  inserted 5000/14292
  inserted 6000/14292
  inserted 7000/14292
  inserted 8000/14292
  inserted 9000/14292
  inserted 10000/14292
  inserted 11000/14292
  inserted 12000/14292
  inserted 13000/14292
  inserted 14000/14292
  inserted 14292/14292
Done. Inserted 14292 rows into jkt_plan.


In [7]:
# Verify: count + show first few inserted rows.
conn = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)
try:
    cur = conn.cursor(dictionary=True)
    cur.execute("SELECT COUNT(*) AS n FROM jkt_plan WHERE plan_id = %s", (PLAN_ID,))
    print("Row count for plan_id", PLAN_ID, ":", cur.fetchone()["n"])
    cur.execute(
        "SELECT * FROM jkt_plan WHERE plan_id = %s ORDER BY date, shift, startTime LIMIT 3",
        (PLAN_ID,),
    )
    for r in cur.fetchall():
        print(r)
finally:
    cur.close()
    conn.close()


Row count for plan_id 1001 : 14292
{'plan_id': '1001', 'skuCode': '1225121Z14112LSTL0', 'skuDescription': '215R14_STEEL KING_10PR_L_TT', 'date': datetime.date(2026, 5, 1), 'shift': 'A', 'startTime': datetime.datetime(2026, 5, 1, 7, 0), 'endTime': datetime.datetime(2026, 5, 1, 15, 0), 'qty': 28, 'cycleTime': 33.0, 'remarks': 'LP Scheduled', 'createdAt': datetime.datetime(2026, 5, 21, 15, 4), 'createdBy': 'Algo8 AI'}
{'plan_id': '1001', 'skuCode': '1325121715100SRBT0', 'skuDescription': '215/75R15 RANGER BRT 100S_TT', 'date': datetime.date(2026, 5, 1), 'shift': 'A', 'startTime': datetime.datetime(2026, 5, 1, 7, 0), 'endTime': datetime.datetime(2026, 5, 1, 15, 0), 'qty': 64, 'cycleTime': 15.0, 'remarks': 'LP Scheduled', 'createdAt': datetime.datetime(2026, 5, 21, 15, 4), 'createdBy': 'Algo8 AI'}
{'plan_id': '1001', 'skuCode': '1325119015008SRBT0', 'skuDescription': '195R15 BRT RANGER TT_107/105S', 'date': datetime.date(2026, 5, 1), 'shift': 'A', 'startTime': datetime.datetime(2026, 5, 1, 

In [8]:
# Compute date-wise capacity utilisation from Shift Schedule.
# Used time = EndTime − StartTime per slot, split at midnight so each date
# only gets its own minutes. Per-machine cap at 1440 (100%). Then average
# across machines that appear on that date.
from collections import defaultdict
from datetime import timedelta

ws = wb["Shift Schedule"]

def split_by_date(start, end):
    cur = start
    while cur < end:
        next_midnight = datetime.combine(cur.date() + timedelta(days=1), datetime.min.time())
        chunk_end = min(end, next_midnight)
        yield cur.date(), (chunk_end - cur).total_seconds() / 60
        cur = chunk_end

busy = defaultdict(float)              # (date, machine) -> minutes used
machines_per_day = defaultdict(set)

for r in range(4, ws.max_row + 1):
    machine = ws.cell(row=r, column=3).value
    s       = ws.cell(row=r, column=5).value
    e       = ws.cell(row=r, column=6).value
    if machine is None or s is None or e is None or e <= s:
        continue
    for d, mins in split_by_date(s, e):
        busy[(d, machine)] += mins
        machines_per_day[d].add(machine)

date_util_pct = {}
for d, machines in machines_per_day.items():
    utils = [min(busy[(d, m)] / 1440, 1.0) for m in machines]
    date_util_pct[d] = round(sum(utils) / len(utils) * 100, 2)

for d in sorted(date_util_pct):
    print(f"  {d}  machines={len(machines_per_day[d]):3d}  util={date_util_pct[d]:6.2f}%")


  2026-05-01  machines=170  util= 70.67%
  2026-05-02  machines=172  util= 97.91%
  2026-05-03  machines=170  util= 99.41%
  2026-05-04  machines=169  util=100.00%
  2026-05-05  machines=174  util= 97.08%
  2026-05-06  machines=169  util= 99.44%
  2026-05-07  machines=166  util= 97.85%
  2026-05-08  machines=162  util= 98.97%
  2026-05-09  machines=159  util=100.00%
  2026-05-10  machines=162  util= 98.10%
  2026-05-11  machines=159  util= 99.27%
  2026-05-12  machines=160  util= 99.00%
  2026-05-13  machines=158  util= 99.65%
  2026-05-14  machines=157  util=100.00%
  2026-05-15  machines=163  util= 93.84%
  2026-05-16  machines=154  util= 98.70%
  2026-05-17  machines=152  util= 99.78%
  2026-05-18  machines=156  util= 96.66%
  2026-05-19  machines=151  util= 99.07%
  2026-05-20  machines=149  util=100.00%
  2026-05-21  machines=157  util= 93.66%
  2026-05-22  machines=148  util= 95.02%
  2026-05-23  machines=148  util= 95.12%
  2026-05-24  machines=153  util= 84.88%
  2026-05-25  ma

In [9]:
# Note: DB column is `creatdBy` (typo) not `createdBy`.
now = datetime.now()
rows_to_insert = [
    (PLAN_ID, d, util_pct, now, CREATED_BY)
    for d, util_pct in sorted(date_util_pct.items())
]

conn = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)
try:
    cur = conn.cursor()
    cur.executemany(
        """INSERT INTO jkt_plan_capacityUtilisation
               (plan_id, date, capacityUtilisation, createdAt, creatdBy)
           VALUES (%s, %s, %s, %s, %s)""",
        rows_to_insert,
    )
    conn.commit()
    print(f"Inserted {cur.rowcount} rows into jkt_plan_capacityUtilisation")
finally:
    cur.close()
    conn.close()


Inserted 32 rows into jkt_plan_capacityUtilisation


In [10]:
conn = mysql.connector.connect(
    host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME,
)
try:
    cur = conn.cursor(dictionary=True)
    cur.execute(
        "SELECT * FROM jkt_plan_capacityUtilisation WHERE plan_id = %s ORDER BY date",
        (PLAN_ID,),
    )
    for r in cur.fetchall():
        print(r)
finally:
    cur.close()
    conn.close()


{'plan_id': '1001', 'date': datetime.date(2026, 5, 1), 'capacityUtilisation': 70.67, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'plan_id': '1001', 'date': datetime.date(2026, 5, 2), 'capacityUtilisation': 97.91, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'plan_id': '1001', 'date': datetime.date(2026, 5, 3), 'capacityUtilisation': 99.41, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'plan_id': '1001', 'date': datetime.date(2026, 5, 4), 'capacityUtilisation': 100.0, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'plan_id': '1001', 'date': datetime.date(2026, 5, 5), 'capacityUtilisation': 97.08, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'plan_id': '1001', 'date': datetime.date(2026, 5, 6), 'capacityUtilisation': 99.44, 'createdAt': datetime.datetime(2026, 5, 21, 15, 34, 20), 'creatdBy': 'Algo8 AI'}
{'pl